# 13. Vectorization and Universal Functions

The main reason to use NumPy is speed, and you only get it if you let NumPy do the looping. This notebook shows why, and the tools for turning loops into array operations.

In [1]:
import numpy as np
import time

rng = np.random.default_rng(0)

## Loops vs vectorized code

A Python loop handles one element at a time, with type checks and function calls on every iteration. A NumPy operation runs a tight compiled loop over the whole buffer.

In [2]:
x = rng.random(1_000_000)

def timeit(fn, repeat=3):
    best = float('inf')
    for _ in range(repeat):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best

def loop_version():
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i] ** 2 + 1
    return out

def numpy_version():
    return x ** 2 + 1

t_loop, t_np = timeit(loop_version), timeit(numpy_version)
print(f'loop:  {t_loop * 1000:.0f} ms')
print(f'numpy: {t_np * 1000:.1f} ms')
print(f'roughly {t_loop / t_np:.0f}x faster')

loop:  244 ms
numpy: 1.2 ms
roughly 198x faster


(Your numbers will differ, but the gap is usually somewhere between 50x and a few hundred x.)

## What's a ufunc?

A **universal function** (ufunc) is a function that works element by element on arrays: `np.add`, `np.sqrt`, `np.maximum`, `np.sin`, and so on. They all support broadcasting, dtype handling and a few extra methods that are worth knowing.

In [3]:
print(type(np.add), np.add.nin, np.add.nout)

<class 'numpy.ufunc'> 2 1


### `reduce`, `accumulate`, `outer`

Every binary ufunc has these methods. `np.sum` is basically `np.add.reduce`, and `np.cumsum` is `np.add.accumulate`, but it works for any ufunc:

In [4]:
a = np.array([3, 1, 4, 1, 5])
print(np.add.reduce(a))
print(np.maximum.accumulate(a))   # running maximum
print(np.multiply.outer([1, 2, 3], [10, 20]))

14
[3 3 4 4 5]
[[10 20]
 [20 40]
 [30 60]]


In [5]:
# running maximum is handy for things like drawdowns
prices = np.array([100, 110, 105, 120, 90, 95])
peak = np.maximum.accumulate(prices)
print('drawdown:', (prices - peak) / peak)

drawdown: [ 0.          0.         -0.04545455  0.         -0.25       -0.20833333]


### `out` and `where`

`out` writes into an existing array (no new allocation). `where` only computes at positions where the mask is true, which is useful for avoiding warnings like division by zero:

In [6]:
num = np.array([1., 2., 3., 4.])
den = np.array([2., 0., 4., 0.])

result = np.zeros_like(num)
np.divide(num, den, out=result, where=den != 0)
result

array([0.5 , 0.  , 0.75, 0.  ])

### `at`: unbuffered in-place operations

With fancy indexing, repeated indices only get applied once when you use `+=`. `np.add.at` applies all of them:

In [7]:
counts = np.zeros(5, dtype=int)
idx = np.array([0, 1, 1, 3, 3, 3])

c1 = counts.copy(); c1[idx] += 1
c2 = counts.copy(); np.add.at(c2, idx, 1)
print(c1)   # repeats collapsed
print(c2)   # actually counted

[1 1 0 1 0]
[1 2 0 3 0]


For counting like this, `np.bincount` is even simpler:

In [8]:
np.bincount(idx, minlength=5)

array([1, 2, 0, 3, 0])

## Turning common loops into array code

### Conditionals → `np.where` / masks

In [9]:
x = rng.normal(size=10)

# loop version
out = []
for v in x:
    out.append(v if v > 0 else 0.1 * v)

# vectorized
vec = np.where(x > 0, x, 0.1 * x)
print(np.allclose(out, vec))

True


### Accumulating in a loop → reductions

In [10]:
values = rng.integers(0, 100, size=1000)

total = 0
for v in values:
    if v % 2 == 0:
        total += v

print(total, values[values % 2 == 0].sum())

24500 24500


### Nested loops → broadcasting

In [11]:
a = rng.random(300)
b = rng.random(400)

def nested():
    out = np.empty((len(a), len(b)))
    for i in range(len(a)):
        for j in range(len(b)):
            out[i, j] = abs(a[i] - b[j])
    return out

def broadcast():
    return np.abs(a[:, None] - b[None, :])

print(np.allclose(nested(), broadcast()))
print(f'nested: {timeit(nested, 1)*1000:.0f} ms, broadcast: {timeit(broadcast)*1000:.2f} ms')

True
nested: 41 ms, broadcast: 0.25 ms


### Grouped sums → `bincount` with weights

In [12]:
group = np.array([0, 2, 1, 0, 2, 2])
amount = np.array([10., 5., 7., 3., 1., 4.])
np.bincount(group, weights=amount)    # total amount per group

array([13.,  7., 10.])

## What about `np.vectorize`?

`np.vectorize` wraps a Python function so it accepts arrays. It's convenient, but it's still a Python loop under the hood, so **it doesn't make anything faster**. Use it for convenience, not performance.

In [13]:
def grade(score):
    if score >= 90: return 'A'
    if score >= 80: return 'B'
    if score >= 70: return 'C'
    return 'F'

scores = rng.integers(50, 100, size=8)
print(scores)
print(np.vectorize(grade)(scores))

[58 88 51 61 53 87 61 81]
['F' 'B' 'F' 'F' 'F' 'B' 'F' 'B']


The truly vectorized version of the same thing:

In [14]:
labels = np.array(['F', 'C', 'B', 'A'])
labels[np.searchsorted([70, 80, 90], scores, side='right')]

array(['F', 'B', 'F', 'F', 'F', 'B', 'F', 'B'], dtype='<U1')

## When you really can't vectorize

Some algorithms depend on the previous step's result in a way that doesn't map to `cumsum`/`accumulate`. In those cases, look at [Numba](https://numba.pydata.org/), which compiles a plain Python loop to machine code, or write that part in Cython.